# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Note: Metadata is an object, not a dict or list

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Publisher: {getattr(metadata, 'author', None)}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")
print(f"License: {getattr(metadata, 'license', None)}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset may contain multiple "record sets", each representing a table of related records. We'll enumerate all record sets and their fields. All references will use their `@id`.


In [ ]:
# Enumerate all record sets and fields using their @id
record_sets = dataset.record_sets

print("Record sets available:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', 'NA')}")
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - Field @id: {field['@id']}, name: {field.get('name', 'NA')}")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll extract all record sets and load them into pandas DataFrames for easy manipulation. Use the record set and field `@id`s from the overview above.

In [ ]:
# List of record set @ids (extracted from previous cell)
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display DataFrame columns for the first available record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in record set ({first_rs_id}): {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records, normalize numeric fields, categorize, remove outliers, etc.

Select a numeric field via its `@id` for detailed analysis. All references must use `@id`.

In [ ]:
# Example: Use the first record set and a numeric field for demonstration
rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes[rs_id] if rs_id else None

if df is not None:
    # Attempt to select a numeric field using its @id (search for candidates)
    numeric_field_id = None
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64']:
            numeric_field_id = col
            break

    # If no numeric field found, fallback to a dummy field
    if numeric_field_id is None:
        numeric_field_id = df.columns[0]

    print(f"Using numeric field @id: {numeric_field_id}")

    # Filter: e.g., numeric_field > threshold
    threshold = 10
    if df[numeric_field_id].dtype in ['float64', 'int64']:
        filtered_df = df[df[numeric_field_id] > threshold]
    else:
        filtered_df = df

    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalizing the numeric field for filtered records
    if df[numeric_field_id].dtype in ['float64', 'int64']:
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a categorical field
    group_field_id = None
    for col in df.columns:
        if df[col].dtype == 'object' and col != numeric_field_id:
            group_field_id = col
            break

    if group_field_id:
        print(f"Grouping filtered records by field @id: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        display(grouped_df.head())
    else:
        print("No categorical group field found.")


## 5. Visualization
Visualize data distributions and relationships between fields represented by their `@id`s.

Below we plot a histogram of the numeric field and, if possible, a barplot of group means.

In [ ]:
import matplotlib.pyplot as plt

if df is not None:
    # Histogram for the numeric field
    if numeric_field_id and df[numeric_field_id].dtype in ['float64', 'int64']:
        plt.figure(figsize=(6,4))
        plt.hist(df[numeric_field_id].dropna(), bins=20, color='skyblue', edgecolor='black')
        plt.title(f"Distribution of numeric field (@id): {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

    # Barplot of group means if available
    if group_field_id:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().dropna()
        group_means.plot(kind='bar', figsize=(8,4), color='coral')
        plt.title(f"Mean of {numeric_field_id} by group (@id): {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()


## 6. Conclusion
This notebook demonstrates how to load and explore a FAIR² dataset using the `mlcroissant` library.

- Record sets, fields, and columns are referenced by their `@id` for reproducibility.
- We loaded the data, performed basic filtering and normalization, grouped and visualized statistics, and identified dataset structure.
- You can extend analysis to additional fields or utilize the Croissant metadata to support further FAIR data workflows.
